# 实践项目 05：空间转录组表达超分辨率

我们将在 Kaggle Notebook 中使用配对的 H&E、低分辨率表达图和高分辨率 Snap25 表达图，训练轻量融合网络，并与插值基线比较。

## 实践任务
1. 核对 he、lr、hr 和 split 四个字段
2. 显示同一区域的 H&E、LR 与 HR
3. 确认训练区、缓冲区和留出区的空间关系
4. 补全 H&E 分支、表达分支与融合模块
5. 完成训练循环并记录验证指标
6. 计算 MAE、相关性和粗网格聚合误差
7. 使用统一 viridis 色标显示预测与误差

## 需要保存的结果
- `task5_data_visualization.png`
- `task5_training_curve.png`
- `task5_prediction_visualization.png`
- `task5_result.json`


## 输出
- `task5_data_visualization.png`
- `task5_training_curve.png`
- `task5_prediction_visualization.png`
- `task5_result.json`


In [ ]:
from pathlib import Path
import json,random
import numpy as np
import matplotlib.pyplot as plt
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from skimage.transform import resize

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/kaggle/working');OUT.mkdir(exist_ok=True)
files=list(Path('/kaggle/input').glob('**/*.npz'))
assert files,'没有找到项目 05 配对 patch NPZ。'
data=np.load(files[0],allow_pickle=True)
print(data.files)

## 任务 1：核对输入字段与空间划分

数据需要包含 `he`、`lr`、`hr` 和 `split`。H&E 形状为 `[N,3,H,W]`，表达为 `[N,1,H,W]`。


In [ ]:
required={'he','lr','hr','split'}
assert required.issubset(data.files),required-set(data.files)
he=data['he'].astype(np.float32);lr=data['lr'].astype(np.float32);hr=data['hr'].astype(np.float32);split=data['split'].astype(str)
# TODO 1：统计各 split 数量、shape、非零比例和最大值
summary=None
print(summary)

In [ ]:
i=np.where(split=='train')[0][0]
fig,ax=plt.subplots(1,4,figsize=(12,3))
ax[0].imshow(np.moveaxis(he[i],0,-1));ax[0].set_title('H&E')
lr_density=lr[i,0]/64.0
ax[1].imshow(lr_density,cmap='viridis');ax[1].set_title('LR density')
ax[2].imshow(hr[i,0],cmap='viridis');ax[2].set_title('HR')
ax[3].imshow(hr[i,0]-lr_density,cmap='viridis');ax[3].set_title('detail')
for a in ax:a.axis('off')
plt.tight_layout();plt.savefig(OUT/'task5_data_visualization.png',dpi=160);plt.show()

In [ ]:
class STDataset(Dataset):
    def __init__(self,kind): self.ids=np.where(split==kind)[0]
    def __len__(self): return len(self.ids)
    def __getitem__(self,k):
        i=self.ids[k]
        lr_total=lr[i]
        lr_density=lr_total/64.0
        x=np.concatenate([he[i],np.log1p(lr_density)],axis=0)
        y=np.log1p(hr[i])
        return torch.from_numpy(x),torch.from_numpy(y),torch.from_numpy(lr_total),i
train_loader=DataLoader(STDataset('train'),batch_size=8,shuffle=True)
val_loader=DataLoader(STDataset('val'),batch_size=8,shuffle=False)
test_loader=DataLoader(STDataset('test'),batch_size=8,shuffle=False)

## 任务 2：补全残差超分辨网络


In [ ]:
class SRNet(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 2：输入 4 通道，输出 1 通道；使用 Softplus 保证非负
        self.body=None
    def forward(self,x):
        lr_log=x[:,3:4]
        return torch.nn.functional.softplus(lr_log+self.body(x))
model=SRNet().to(DEVICE)
print(sum(p.numel() for p in model.parameters()))

## 任务 3：损失与训练

损失由 log 空间 L1 与聚合一致性组成。聚合一致性把预测按 8×8 区域求和后与 LR 对比。


In [ ]:
def aggregate8(x): return torch.nn.functional.avg_pool2d(x,8,8)*64

def loss_fn(pred_log,target_log,lr_raw):
    pred=torch.expm1(pred_log).clamp_min(0);target=torch.expm1(target_log).clamp_min(0)
    l1=(pred_log-target_log).abs().mean()
    # TODO 3：aggregate8(pred) 与 avg_pool2d(lr_raw,8,8) 应表示同一粗尺度总量
    consistency=None
    return l1+.1*consistency

opt=torch.optim.Adam(model.parameters(),lr=1e-3)
def run(loader,training):
    model.train(training);ls=[];maes=[]
    for x,y,lr_total,_ in loader:
        x=x.to(DEVICE);y=y.to(DEVICE);lr_total=lr_total.to(DEVICE)
        if training: opt.zero_grad()
        pred=model(x);loss=loss_fn(pred,y,lr_total)
        if training: loss.backward();opt.step()
        ls.append(float(loss.detach().cpu()));maes.append(float((torch.expm1(pred).clamp_min(0)-torch.expm1(y)).abs().mean().detach().cpu()))
    return np.mean(ls),np.mean(maes)

history=[];best=None;best_mae=1e9
for epoch in range(5):
    tl,tm=run(train_loader,True);vl,vm=run(val_loader,False);history.append((tl,tm,vl,vm));print(epoch+1,history[-1])
    if vm<best_mae:best_mae=vm;best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best)

In [ ]:
h=np.array(history);fig,ax=plt.subplots(1,2,figsize=(9,3.5));ax[0].plot(h[:,0],label='train');ax[0].plot(h[:,2],label='validation');ax[0].set_title('loss');ax[0].legend();ax[1].plot(h[:,1],label='train');ax[1].plot(h[:,3],label='validation');ax[1].set_title('MAE');ax[1].legend();plt.tight_layout();plt.savefig(OUT/'task5_training_curve.png',dpi=160);plt.show()

## 任务 4：与插值基线比较


In [ ]:
model.eval();mae_model=[];mae_base=[];corr_model=[];examples=[]
with torch.no_grad():
    for x,y,lr_total,ids in test_loader:
        pred=torch.expm1(model(x.to(DEVICE))).clamp_min(0).cpu().numpy();target=torch.expm1(y).numpy();base=torch.expm1(x[:,3:4]).numpy()
        for k in range(len(x)):
            mae_model.append(np.abs(pred[k]-target[k]).mean());mae_base.append(np.abs(base[k]-target[k]).mean())
            corr_model.append(np.corrcoef(pred[k].ravel(),target[k].ravel())[0,1])
            if len(examples)<3:examples.append((np.moveaxis(x[k,:3].numpy(),0,-1),base[k,0],target[k,0],pred[k,0]))

fig,ax=plt.subplots(len(examples),5,figsize=(14,3*len(examples)))
for r,(im,b,t,p) in enumerate(examples):
    vmax=np.percentile(t,99)
    for c,(arr,title) in enumerate([(im,'H&E'),(b,'LR baseline'),(t,'HR true'),(p,'model'),(np.abs(p-t),'absolute error')]):
        if c==0:ax[r,c].imshow(arr)
        else:ax[r,c].imshow(arr,cmap='viridis',vmin=0,vmax=vmax if c<4 else None)
        ax[r,c].set_title(title);ax[r,c].axis('off')
plt.tight_layout();plt.savefig(OUT/'task5_prediction_visualization.png',dpi=160);plt.show()

result={'test_mae_model':float(np.mean(mae_model)),'test_mae_interpolation':float(np.mean(mae_base)),'test_pearson_model':float(np.nanmean(corr_model)),'train_patches':len(STDataset('train')),'test_patches':len(STDataset('test')),'seed':SEED}
(OUT/'task5_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8');result

## 任务 5：结论

比较模型与插值基线，检查误差集中区域，并说明同一切片受控重建能够支持的结论范围。
